In [36]:
import boto3

bucket_name = "cartwave"
prefix = ""

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if "Contents" in response:
    print("Files found in bucket:")
    for obj in response["Contents"]:
        print(obj["Key"])
else:
    print("No files found.")

Files found in bucket:
raw/
raw/olist_customers_dataset.csv
raw/olist_order_items_dataset.csv
raw/olist_order_payments_dataset.csv
raw/olist_orders_dataset.csv


# 1. Import Libraries and Define File Paths

In this section, we import the Python libraries required for data loading, cleaning, feature engineering, visualization, and model preparation. We also define the file paths for the raw Olist CSV datasets that will be used in this project.

These datasets represent different aspects of the e-commerce order lifecycle, including orders, customers, payments, and order items. Since the final machine learning model will predict delayed delivery at the order level, these separate files will later be merged into one analytical dataset.

In [37]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Optional visualization libraries for EDA later
import matplotlib.pyplot as plt

# 2. Load the Raw Datasets

In this section, we read the raw CSV files into pandas DataFrames. Each file captures a different layer of the business process:

- Orders dataset: order timestamps and delivery status
- Order items dataset: item-level price and freight information
- Payments dataset: payment type, installments, and payment value
- Customers dataset: customer location and customer identifiers

Loading these datasets separately allows us to inspect their shapes, columns, and quality before merging them into one modeling dataset.

In [38]:
import boto3
import pandas as pd
from io import BytesIO

bucket_name = "cartwave"
s3 = boto3.client("s3")

def read_csv_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()))

orders_df = read_csv_from_s3(bucket_name, "raw/olist_orders_dataset.csv")
items_df = read_csv_from_s3(bucket_name, "raw/olist_order_items_dataset.csv")
payments_df = read_csv_from_s3(bucket_name, "raw/olist_order_payments_dataset.csv")
customers_df = read_csv_from_s3(bucket_name, "raw/olist_customers_dataset.csv")

print("Orders shape:", orders_df.shape)
print("Items shape:", items_df.shape)
print("Payments shape:", payments_df.shape)
print("Customers shape:", customers_df.shape)

Orders shape: (99441, 8)
Items shape: (112650, 7)
Payments shape: (103886, 5)
Customers shape: (99441, 5)


# 3. Inspect the Raw Datasets

Before performing any preprocessing or merging, it is important to understand the structure of each dataset. This includes reviewing the first few rows, the shape of each table, the column names, and the data types.

This step helps confirm that the expected keys such as `order_id` and `customer_id` are present and also provides an initial understanding of the variables available for later feature engineering and model development.

In [41]:
# Display shapes
print("Orders shape:", orders_df.shape)
print("Items shape:", items_df.shape)
print("Payments shape:", payments_df.shape)
print("Customers shape:", customers_df.shape)

print("\nOrders columns:")
print(orders_df.columns.tolist())

print("\nItems columns:")
print(items_df.columns.tolist())

print("\nPayments columns:")
print(payments_df.columns.tolist())

print("\nCustomers columns:")
print(customers_df.columns.tolist())

# Preview first 5 rows
print("\nOrders preview:")
print(orders_df.head())

print("\nItems preview:")
print(items_df.head())

print("\nPayments preview:")
print(payments_df.head())

print("\nCustomers preview:")
print(customers_df.head())

# Data types
print("\nOrders data types:")
print(orders_df.dtypes)

print("\nItems data types:")
print(items_df.dtypes)

print("\nPayments data types:")
print(payments_df.dtypes)

print("\nCustomers data types:")
print(customers_df.dtypes)

Orders shape: (99441, 8)
Items shape: (112650, 7)
Payments shape: (103886, 5)
Customers shape: (99441, 5)

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Customers columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Orders preview:
                           order_id  ... order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  ...           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  ...           2018-08-13 00:00:00
2  47770eb9100c2d0c44946d9cf07ec65d  ...           2018-09-04 00:00:00
3  949d5b44dbf5de918fe9c16f97b4

# 4. Check for Missing Values and Duplicate Records

Data quality checks are essential before building a modeling dataset. In this section, missing values and duplicate records are examined for each raw dataset.

This step helps identify potential issues that may affect later preprocessing, such as null delivery dates, incomplete payment information, or duplicated rows that could distort aggregations and model training results.

In [42]:
def missing_summary(df, name):
    print(f"\nMissing values summary for {name}:")
    print(df.isnull().sum().sort_values(ascending=False))

def duplicate_summary(df, name):
    print(f"\nDuplicate rows in {name}: {df.duplicated().sum()}")

missing_summary(orders_df, "orders_df")
missing_summary(items_df, "items_df")
missing_summary(payments_df, "payments_df")
missing_summary(customers_df, "customers_df")

duplicate_summary(orders_df, "orders_df")
duplicate_summary(items_df, "items_df")
duplicate_summary(payments_df, "payments_df")
duplicate_summary(customers_df, "customers_df")


Missing values summary for orders_df:
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

Missing values summary for items_df:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Missing values summary for payments_df:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Missing values summary for customers_df:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows in orders_df:

# 5. Convert Date Columns to Datetime Format

The orders dataset contains several timestamp columns related to order purchase, approval, shipping, and delivery. These columns must be converted into datetime format so that time-based calculations can be performed accurately.

This conversion is necessary for creating the target variable and for deriving useful features such as purchase month, purchase hour, and delivery-related durations.

In [43]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col], errors="coerce")

print(orders_df[date_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


# 6. Filter the Dataset to Delivered Orders

For the first version of the late delivery prediction model, only delivered orders are retained. This is because the target variable requires both the actual customer delivery date and the estimated delivery date.

Orders that were canceled, unavailable, or not yet delivered are excluded because the final delivery outcome is not available for those records.

In [44]:
delivered_orders_df = orders_df[
    (orders_df["order_status"] == "delivered") &
    (orders_df["order_delivered_customer_date"].notna()) &
    (orders_df["order_estimated_delivery_date"].notna())
].copy()

print("Delivered orders shape:", delivered_orders_df.shape)
print("\nOrder status counts in filtered data:")
print(delivered_orders_df["order_status"].value_counts())

Delivered orders shape: (96470, 8)

Order status counts in filtered data:
delivered    96470
Name: order_status, dtype: int64


# 7. Create the Target Variable

The business objective is to predict whether an order will be delivered late. To represent this as a machine learning target, a binary variable called `late_delivery_flag` is created.

The target is defined as follows:

- `1` if the actual delivery date is later than the estimated delivery date
- `0` if the order is delivered on time or earlier than expected

This transforms the business problem into a binary classification problem.

In [46]:
delivered_orders_df["late_delivery_flag"] = (
    delivered_orders_df["order_delivered_customer_date"] >
    delivered_orders_df["order_estimated_delivery_date"]
).astype(int)

print(delivered_orders_df[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "late_delivery_flag"
]].head())

print("\nTarget distribution:")
print(delivered_orders_df["late_delivery_flag"].value_counts(normalize=True))

                           order_id  ... late_delivery_flag
0  e481f51cbdc54678b7cc49136f2d6af7  ...                  0
1  53cdb2fc8bc7dce0b6741e2150273451  ...                  0
2  47770eb9100c2d0c44946d9cf07ec65d  ...                  0
3  949d5b44dbf5de918fe9c16f97b45f8a  ...                  0
4  ad21c59c0840e6cb83a9ceb5573f8159  ...                  0

[5 rows x 4 columns]

Target distribution:
0    0.918876
1    0.081124
Name: late_delivery_flag, dtype: float64


# 8. Aggregate Order Item Data to the Order Level

The order items dataset contains one row per item, which means an order may appear multiple times. Since the modeling dataset should contain one row per order, the item-level data must be aggregated.

This section creates order-level item features, including the number of items, total item price, total freight value, average item price, and number of unique sellers involved in each order.

In [47]:
items_agg = items_df.groupby("order_id").agg(
    num_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
    num_unique_sellers=("seller_id", "nunique")
).reset_index()

print("Aggregated items shape:", items_agg.shape)
print(items_agg.head())

Aggregated items shape: (98666, 6)
                           order_id  ...  num_unique_sellers
0  00010242fe8c5a6d1ba2dd792cb16214  ...                   1
1  00018f77f2f0320c557190d7a144bdd3  ...                   1
2  000229ec398224ef6ca0657da4fc703e  ...                   1
3  00024acbcdf0a6daa1e931b038114c75  ...                   1
4  00042b26cf59d7ce69dfabb4e55b4fd9  ...                   1

[5 rows x 6 columns]


# 9. Aggregate Payment Data to the Order Level

The payments dataset may contain multiple payment records per order. To align this table with the order-level modeling dataset, payment information must also be aggregated.

This section creates payment-related order-level features such as the number of payment records, total payment value, maximum number of installments, and the main payment type used for the order.

In [48]:
import numpy as np

payment_type_mode = (
    payments_df.groupby("order_id")["payment_type"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .reset_index()
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = payments_df.groupby("order_id").agg(
    num_payment_records=("payment_sequential", "count"),
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max")
).reset_index()

payments_agg = payments_agg.merge(payment_type_mode, on="order_id", how="left")

print("Aggregated payments shape:", payments_agg.shape)
print(payments_agg.head())

Aggregated payments shape: (99440, 5)
                           order_id  ...  main_payment_type
0  00010242fe8c5a6d1ba2dd792cb16214  ...        credit_card
1  00018f77f2f0320c557190d7a144bdd3  ...        credit_card
2  000229ec398224ef6ca0657da4fc703e  ...        credit_card
3  00024acbcdf0a6daa1e931b038114c75  ...        credit_card
4  00042b26cf59d7ce69dfabb4e55b4fd9  ...        credit_card

[5 rows x 5 columns]


# 10. Select Customer Features

The customers dataset contains customer identifiers and geographic information. Customer location may be useful for late delivery prediction because shipping performance can vary by region.

In this section, the customer fields most relevant to the first model are retained, including customer city and customer state.

In [49]:
customers_selected = customers_df[[
    "customer_id",
    "customer_city",
    "customer_state"
]].copy()

print("Selected customer dataset shape:", customers_selected.shape)
print(customers_selected.head())

Selected customer dataset shape: (99441, 3)
                        customer_id          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  sao bernardo do campo             SP
2  4e7b3e00288586ebd08712fdd0374a03              sao paulo             SP
3  b2b6027bc5c5109e529d4dc6358b12c3        mogi das cruzes             SP
4  4f2d8ab171c80ec8364f7c12e35b23ad               campinas             SP


# 11. Merge the Datasets into a Single Modeling Table

The delivered orders dataset is merged with the aggregated order item data, aggregated payment data, and selected customer data.

The resulting dataset contains one row per delivered order and combines delivery outcome information with item-level, payment-level, and customer-level features. This merged dataset serves as the basis for feature engineering and model training.

In [50]:
merged_df = delivered_orders_df.merge(items_agg, on="order_id", how="left")
merged_df = merged_df.merge(payments_agg, on="order_id", how="left")
merged_df = merged_df.merge(customers_selected, on="customer_id", how="left")

print("Merged dataset shape:", merged_df.shape)
print(merged_df.head())

Merged dataset shape: (96470, 20)
                           order_id  ... customer_state
0  e481f51cbdc54678b7cc49136f2d6af7  ...             SP
1  53cdb2fc8bc7dce0b6741e2150273451  ...             BA
2  47770eb9100c2d0c44946d9cf07ec65d  ...             GO
3  949d5b44dbf5de918fe9c16f97b45f8a  ...             RN
4  ad21c59c0840e6cb83a9ceb5573f8159  ...             SP

[5 rows x 20 columns]


# 12. Create Time-Based Features

Time-based features can provide useful signals for predicting late deliveries. In this section, several derived features are created from the order timestamps.

These include purchase year, month, day of week, hour of purchase, approval delay in hours, and estimated delivery time in days. These variables help convert raw timestamps into interpretable numerical features for model training.

In [51]:
merged_df["purchase_year"] = merged_df["order_purchase_timestamp"].dt.year
merged_df["purchase_month"] = merged_df["order_purchase_timestamp"].dt.month
merged_df["purchase_dayofweek"] = merged_df["order_purchase_timestamp"].dt.dayofweek
merged_df["purchase_hour"] = merged_df["order_purchase_timestamp"].dt.hour

merged_df["approval_delay_hours"] = (
    (merged_df["order_approved_at"] - merged_df["order_purchase_timestamp"])
    .dt.total_seconds() / 3600
)

merged_df["estimated_delivery_days"] = (
    (merged_df["order_estimated_delivery_date"] - merged_df["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
)

print(merged_df[[
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days"
]].head())

   purchase_year  purchase_month  ...  approval_delay_hours  estimated_delivery_days
0           2017              10  ...              0.178333                15.544063
1           2018               7  ...             30.713889                19.137766
2           2018               8  ...              0.276111                26.639711
3           2017              11  ...              0.298056                26.188819
4           2018               2  ...              1.030556                12.112049

[5 rows x 6 columns]


# 13. Select Final Modeling Features

Not all columns in the merged dataset should be used directly for model training. Some fields are identifiers, while others may contain information that would not be available at prediction time.

This section selects the features that are most appropriate for the first version of the late delivery prediction model while excluding non-predictive or leakage-prone variables such as order identifiers and actual delivery timestamps.

In [52]:
model_df = merged_df[[
    "late_delivery_flag",
    "num_items",
    "total_price",
    "total_freight",
    "avg_item_price",
    "num_unique_sellers",
    "num_payment_records",
    "total_payment_value",
    "max_installments",
    "main_payment_type",
    "customer_city",
    "customer_state",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days"
]].copy()

print("Model dataset shape:", model_df.shape)
print(model_df.head())

Model dataset shape: (96470, 17)
   late_delivery_flag  num_items  ...  approval_delay_hours  estimated_delivery_days
0                   0          1  ...              0.178333                15.544063
1                   0          1  ...             30.713889                19.137766
2                   0          1  ...              0.276111                26.639711
3                   0          1  ...              0.298056                26.188819
4                   0          1  ...              1.030556                12.112049

[5 rows x 17 columns]


# 14. Handle Missing Values

After merging multiple source tables, some missing values may remain. Machine learning models require complete input data, so missing values must be handled before training.

In this section, missing numerical values are imputed using the median of each column, while missing categorical values are replaced with the label `Unknown`.

In [53]:
numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = model_df.select_dtypes(include=["object"]).columns.tolist()

for col in numeric_cols:
    if col != "late_delivery_flag":
        model_df[col] = model_df[col].fillna(model_df[col].median())

for col in categorical_cols:
    model_df[col] = model_df[col].fillna("Unknown")

print("Missing values after treatment:")
print(model_df.isnull().sum())

Missing values after treatment:
late_delivery_flag         0
num_items                  0
total_price                0
total_freight              0
avg_item_price             0
num_unique_sellers         0
num_payment_records        0
total_payment_value        0
max_installments           0
main_payment_type          0
customer_city              0
customer_state             0
purchase_month             0
purchase_dayofweek         0
purchase_hour              0
approval_delay_hours       0
estimated_delivery_days    0
dtype: int64


# 15. Encode Categorical Variables

The modeling dataset contains categorical variables such as payment type, customer city, and customer state. Since most machine learning algorithms require numerical input, these categorical features must be encoded.

For the first version of the model, label encoding is applied as a simple and efficient approach. More advanced encoding methods can be explored in future iterations if needed.

In [54]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    label_encoders[col] = le

print(model_df.head())

   late_delivery_flag  num_items  ...  approval_delay_hours  estimated_delivery_days
0                   0          1  ...              0.178333                15.544063
1                   0          1  ...             30.713889                19.137766
2                   0          1  ...              0.276111                26.639711
3                   0          1  ...              0.298056                26.188819
4                   0          1  ...              1.030556                12.112049

[5 rows x 17 columns]


# 16. Split the Data into Training and Testing Sets

To evaluate model performance properly, the modeling dataset is divided into training and testing subsets. The training set is used to fit the machine learning model, while the testing set is used to assess how well the model generalizes to unseen data.

A stratified split is used so that the proportion of late and on-time deliveries remains similar in both subsets.

In [55]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["late_delivery_flag"])
y = model_df["late_delivery_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_test distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (77176, 16)
X_test shape: (19294, 16)
y_train distribution:
0    0.918874
1    0.081126
Name: late_delivery_flag, dtype: float64
y_test distribution:
0    0.918887
1    0.081113
Name: late_delivery_flag, dtype: float64


# 17. Save the Processed Training and Testing Datasets

After preprocessing and splitting the data, the final training and testing datasets are saved as CSV files. These files can later be reused for model training, validation, and reproducibility.

For many cloud-based training workflows, it is helpful to save the target column as the first column in the final CSV output.

In [56]:
import os

processed_dir = "/tmp/data/processed"
os.makedirs(processed_dir, exist_ok=True)

train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([y_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

merged_output_path = os.path.join(processed_dir, "merged_orders.csv")
train_output_path = os.path.join(processed_dir, "train.csv")
test_output_path = os.path.join(processed_dir, "test.csv")

model_df.to_csv(merged_output_path, index=False)
train_df.to_csv(train_output_path, index=False, header=False)
test_df.to_csv(test_output_path, index=False, header=False)

print("Saved files:")
print(merged_output_path)
print(train_output_path)
print(test_output_path)
print("\nProcessed folder contents:")
print(os.listdir(processed_dir))

Saved files:
/tmp/data/processed/merged_orders.csv
/tmp/data/processed/train.csv
/tmp/data/processed/test.csv

Processed folder contents:
['merged_orders.csv', 'train.csv', 'test.csv']


# 18. Upload Processed Files to Amazon S3

After preparing the merged, training, and testing datasets locally, the files can be uploaded back to Amazon S3. This supports cloud-based model training workflows and keeps the processed data centrally accessible for the project team.

In [57]:
import boto3

bucket_name = "cartwave"
s3 = boto3.client("s3")

def upload_file_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)
    print(f"Uploaded {local_path} to s3://{bucket}/{key}")

upload_file_to_s3(merged_output_path, bucket_name, "processed/merged_orders.csv")
upload_file_to_s3(train_output_path, bucket_name, "processed/train.csv")
upload_file_to_s3(test_output_path, bucket_name, "processed/test.csv")

Uploaded /tmp/data/processed/merged_orders.csv to s3://cartwave/processed/merged_orders.csv
Uploaded /tmp/data/processed/train.csv to s3://cartwave/processed/train.csv
Uploaded /tmp/data/processed/test.csv to s3://cartwave/processed/test.csv


## Summary of Preprocessing

At this stage, the raw Olist datasets have been inspected, filtered, aggregated, and merged into a single order-level modeling dataset. A binary target variable for late delivery prediction has been created, relevant features have been engineered, missing values have been handled, and categorical variables have been encoded. Finally, the data has been split into training and testing sets and saved for the next modeling stage.